<a href="https://colab.research.google.com/github/SakshyamAryal077/FOUNDATION-MODEL-DOWNSTREAM-PREDICTION/blob/main/Hard_Negative_Sampling_OMI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 2 — Hard-Negative Sampling for OMI Detection

This notebook keeps the selected Task-1 ECGFounder baseline unchanged and changes only the **negative training distribution**.

Hard negative:
- `OMI == 0`
- and (`STEMI == 1` or `NSTEMI == 1`)

Easy negative:
- `OMI == 0`
- `STEMI == 0`
- `NSTEMI == 0`

The sampler preserves the overall positive-vs-negative sampling mass while increasing the proportion of hard negatives among sampled negatives.


In [1]:
# ============================================================
# ECGFOUNDER — MODEL SETUP
# CELL 1: DOWNLOAD PRETRAINED 12-LEAD MODEL
# ============================================================

import os
import requests
from tqdm.auto import tqdm
from google.colab import drive
import requests
import os
from tqdm.auto import tqdm
drive.mount("/content/drive")

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

PROJECT_DIR = "/content/drive/MyDrive/ECG_Project"

MODEL_DIR = os.path.join(
    PROJECT_DIR,
    "ECGFounder"
)

MODEL_PATH = os.path.join(
    MODEL_DIR,
    "12_lead_ECGFounder.pth"
)

os.makedirs(MODEL_DIR, exist_ok=True)

# Official Hugging Face download URL
MODEL_URL = (
    "https://huggingface.co/PKUDigitalHealth/ECGFounder/"
    "resolve/main/12_lead_ECGFounder.pth"
)

Mounted at /content/drive


In [2]:
import os

PROJECT_DIR = "/content/drive/MyDrive/ECG_Project"

for root, dirs, files in os.walk(PROJECT_DIR):
    for file in files:
        if "ECGFounder" in file or file.endswith(".pth"):
            print(os.path.join(root, file))

/content/drive/MyDrive/ECG_Project/ECGFounder/12_lead_ECGFounder.pth
/content/drive/MyDrive/ECG_Project/results/experiments/stage1_frozen_backbone/20260829_092620/best.pth
/content/drive/MyDrive/ECG_Project/results/experiments/stage1_frozen_backbone/20260829_103039/best.pth
/content/drive/MyDrive/ECG_Project/results/experiments/stage1_frozen_backbone/20260829_105222/best.pth
/content/drive/MyDrive/ECG_Project/results/experiments/stage1_frozen_backbone/20260829_110136/best.pth
/content/drive/MyDrive/ECG_Project/results/experiments/stage1_frozen_backbone/20260829_113850/best.pth
/content/drive/MyDrive/ECG_Project/results/experiments/stage1_frozen_backbone/20260829_113850/epoch_011.pth
/content/drive/MyDrive/ECG_Project/results/experiments/stage1_frozen_backbone/20260829_113850/epoch_012.pth
/content/drive/MyDrive/ECG_Project/results/experiments/stage1_frozen_backbone/20260829_113850/epoch_013.pth
/content/drive/MyDrive/ECG_Project/results/experiments/stage1_frozen_backbone/20260829_11385

In [3]:
# CELL 1 — IMPORTS, PATHS, DEVICE

import os
import random
import h5py
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, confusion_matrix
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PROJECT_DIR = "/content/drive/MyDrive/ECG_Project"
PRETRAINED_PATH = os.path.join(PROJECT_DIR, "ECGFounder", "12_lead_ECGFounder.pth")

BASE = os.path.join(PROJECT_DIR, "results", "preprocessing", "processed")
TRAIN_H5 = f"{BASE}/train_fixed.h5"
VAL_H5   = f"{BASE}/validation.h5"
TRAIN_CSV = f"{BASE}/train_omi.csv"
VAL_CSV   = f"{BASE}/validation_omi.csv"

SAVE_DIR = os.path.join(PROJECT_DIR, "OMI_model", "task2_hard_negative")
os.makedirs(SAVE_DIR, exist_ok=True)

print("Device:", DEVICE)
print("Save directory:", SAVE_DIR)


Device: cuda
Save directory: /content/drive/MyDrive/ECG_Project/OMI_model/task2_hard_negative


## Before running
Set `baseline_mode` to the Task-1 winner:
- `"full"` if Full FT wins
- `"last2"` if Last2 wins

Task 2 must start from the **original pretrained ECGFounder**, not from the Task-1 OMI checkpoint.


In [4]:
# CELL 2 — CONFIG

CONFIG = {
    "baseline_mode": "last2",   # change to "last2" only if Last2 wins Task 1

    "head_lr": 1e-4,
    "backbone_lr": 1e-5,
    "early_backbone_lr": 1e-6,

    "weight_decay": 1e-4,
    "epochs": 30,
    "patience": 5,
    "max_grad_norm": 5.0,
    "batch_size": 16,

    # Share of hard negatives among sampled negatives.
    # Overall positive-vs-negative sampling mass is preserved.
    "target_hard_negative_fraction": 0.50,

    "save_dir": SAVE_DIR,
}

print(CONFIG)


{'baseline_mode': 'last2', 'head_lr': 0.0001, 'backbone_lr': 1e-05, 'early_backbone_lr': 1e-06, 'weight_decay': 0.0001, 'epochs': 30, 'patience': 5, 'max_grad_norm': 5.0, 'batch_size': 16, 'target_hard_negative_fraction': 0.5, 'save_dir': '/content/drive/MyDrive/ECG_Project/OMI_model/task2_hard_negative'}


In [5]:
# CELL 3 — ECGFOUNDER ARCHITECTURE

class MyConv1dPadSame(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, groups=1):
        super().__init__()
        self.kernel_size = kernel_size
        self.stride = stride
        self.conv = nn.Conv1d(
            in_channels, out_channels,
            kernel_size=kernel_size,
            stride=stride,
            groups=groups
        )

    def forward(self, x):
        in_dim = x.shape[-1]
        out_dim = (in_dim + self.stride - 1) // self.stride
        p = max(0, (out_dim - 1) * self.stride + self.kernel_size - in_dim)
        left = p // 2
        right = p - left
        x = F.pad(x, (left, right), "constant", 0)
        return self.conv(x)


class MyMaxPool1dPadSame(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.kernel_size = kernel_size
        self.max_pool = nn.MaxPool1d(kernel_size=kernel_size)

    def forward(self, x):
        p = max(0, self.kernel_size - 1)
        left = p // 2
        right = p - left
        x = F.pad(x, (left, right), "constant", 0)
        return self.max_pool(x)


class Swish(nn.Module):
    def forward(self, x):
        return x * torch.sigmoid(x)


class BasicBlock(nn.Module):
    def __init__(
        self, in_channels, out_channels, ratio, kernel_size,
        stride, groups, downsample, is_first_block=False,
        use_bn=True, use_do=True
    ):
        super().__init__()

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.downsample = downsample
        self.stride = stride if downsample else 1
        self.is_first_block = is_first_block
        self.use_bn = use_bn
        self.use_do = use_do

        middle_channels = int(out_channels * ratio)

        self.bn1 = nn.BatchNorm1d(in_channels)
        self.activation1 = Swish()
        self.do1 = nn.Dropout(p=0.5)
        self.conv1 = MyConv1dPadSame(in_channels, middle_channels, 1, 1, 1)

        self.bn2 = nn.BatchNorm1d(middle_channels)
        self.activation2 = Swish()
        self.do2 = nn.Dropout(p=0.5)
        self.conv2 = MyConv1dPadSame(
            middle_channels, middle_channels,
            kernel_size, self.stride, groups
        )

        self.bn3 = nn.BatchNorm1d(middle_channels)
        self.activation3 = Swish()
        self.do3 = nn.Dropout(p=0.5)
        self.conv3 = MyConv1dPadSame(middle_channels, out_channels, 1, 1, 1)

        r = 2
        self.se_fc1 = nn.Linear(out_channels, out_channels // r)
        self.se_fc2 = nn.Linear(out_channels // r, out_channels)
        self.se_activation = Swish()

        if downsample:
            self.max_pool = MyMaxPool1dPadSame(kernel_size=self.stride)

    def forward(self, x):
        identity = x
        out = x

        if not self.is_first_block:
            if self.use_bn:
                out = self.bn1(out)
            out = self.activation1(out)
            if self.use_do:
                out = self.do1(out)

        out = self.conv1(out)

        if self.use_bn:
            out = self.bn2(out)
        out = self.activation2(out)
        if self.use_do:
            out = self.do2(out)

        out = self.conv2(out)

        if self.use_bn:
            out = self.bn3(out)
        out = self.activation3(out)
        if self.use_do:
            out = self.do3(out)

        out = self.conv3(out)

        se = out.mean(-1)
        se = self.se_fc1(se)
        se = self.se_activation(se)
        se = self.se_fc2(se)
        se = torch.sigmoid(se)

        out = torch.einsum("abc,ab->abc", out, se)

        if self.downsample:
            identity = self.max_pool(identity)

        if self.out_channels != self.in_channels:
            identity = identity.transpose(-1, -2)
            ch1 = (self.out_channels - self.in_channels) // 2
            ch2 = self.out_channels - self.in_channels - ch1
            identity = F.pad(identity, (ch1, ch2), "constant", 0)
            identity = identity.transpose(-1, -2)

        out += identity
        return out


class BasicStage(nn.Module):
    def __init__(
        self, in_channels, out_channels, ratio, kernel_size,
        stride, groups, i_stage, m_blocks,
        use_bn=True, use_do=True
    ):
        super().__init__()
        self.block_list = nn.ModuleList()

        for i_block in range(m_blocks):
            is_first_block = (i_stage == 0 and i_block == 0)
            downsample = (i_block == 0)
            block_stride = stride if downsample else 1
            block_in = in_channels if i_block == 0 else out_channels

            self.block_list.append(
                BasicBlock(
                    in_channels=block_in,
                    out_channels=out_channels,
                    ratio=ratio,
                    kernel_size=kernel_size,
                    stride=block_stride,
                    groups=groups,
                    downsample=downsample,
                    is_first_block=is_first_block,
                    use_bn=use_bn,
                    use_do=use_do
                )
            )

    def forward(self, x):
        for block in self.block_list:
            x = block(x)
        return x


class Net1D(nn.Module):
    def __init__(
        self, in_channels, base_filters, ratio,
        filter_list, m_blocks_list,
        kernel_size, stride, groups_width,
        n_classes, use_bn=True, use_do=True,
        return_features=False
    ):
        super().__init__()

        self.use_bn = use_bn
        self.return_features = return_features
        self.n_stages = len(filter_list)

        self.first_conv = MyConv1dPadSame(
            in_channels, base_filters, kernel_size, 2
        )
        self.first_bn = nn.BatchNorm1d(base_filters)
        self.first_activation = Swish()

        self.stage_list = nn.ModuleList()
        current_in = base_filters

        for i_stage, out_channels in enumerate(filter_list):
            self.stage_list.append(
                BasicStage(
                    in_channels=current_in,
                    out_channels=out_channels,
                    ratio=ratio,
                    kernel_size=kernel_size,
                    stride=stride,
                    groups=out_channels // groups_width,
                    i_stage=i_stage,
                    m_blocks=m_blocks_list[i_stage],
                    use_bn=use_bn,
                    use_do=use_do
                )
            )
            current_in = out_channels

        self.dense = nn.Linear(current_in, n_classes)

    def forward(self, x):
        x = self.first_conv(x)

        if self.use_bn:
            x = self.first_bn(x)

        x = self.first_activation(x)

        for stage in self.stage_list:
            x = stage(x)

        deep_features = x.mean(-1)
        out = self.dense(deep_features)

        if self.return_features:
            return out, deep_features

        return out


In [6]:
# CELL 4 — LOAD ORIGINAL PRETRAINED MODEL + FRESH OMI HEAD

model = Net1D(
    in_channels=12,
    base_filters=64,
    ratio=1,
    filter_list=[64, 160, 160, 400, 400, 1024, 1024],
    m_blocks_list=[2, 2, 2, 3, 3, 4, 4],
    kernel_size=16,
    stride=2,
    groups_width=16,
    n_classes=150,
    use_bn=False,
    use_do=False,
    return_features=True
)

checkpoint = torch.load(
    PRETRAINED_PATH,
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(checkpoint["state_dict"])

model.dense = nn.Linear(1024, 1)
model.return_features = False
model = model.to(DEVICE)

print("Original ECGFounder loaded.")
print("Fresh OMI head:", model.dense)


Original ECGFounder loaded.
Fresh OMI head: Linear(in_features=1024, out_features=1, bias=True)


In [7]:
# CELL 5 — DATASET

class OMIDataset(Dataset):
    def __init__(self, h5_path, csv_path):
        self.h5_path = h5_path
        self.df = pd.read_csv(csv_path)

        with h5py.File(self.h5_path, "r") as f:
            ids = f["record_ids"][:]
            self.record_ids = [
                x.decode() if isinstance(x, bytes) else str(x)
                for x in ids
            ]
            self.length = len(f["signals"])

        assert self.length == len(self.df)

        csv_ids = self.df["record_id"].astype(str).str.zfill(5).tolist()
        h5_ids = [str(x).zfill(5) for x in self.record_ids]
        assert csv_ids == h5_ids, "CSV and H5 order mismatch."

        self.labels = self.df["OMI"].astype(np.float32).values
        self.h5 = None

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        if self.h5 is None:
            self.h5 = h5py.File(self.h5_path, "r")

        ecg = torch.tensor(self.h5["signals"][idx], dtype=torch.float32)
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        return ecg, label


train_dataset = OMIDataset(TRAIN_H5, TRAIN_CSV)
val_dataset = OMIDataset(VAL_H5, VAL_CSV)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))


Train: 15273
Validation: 2685


In [8]:
# CELL 6 — HARD-NEGATIVE GROUPS

train_df = train_dataset.df.copy()

required = ["OMI", "STEMI", "NSTEMI"]
missing = [c for c in required if c not in train_df.columns]

if missing:
    raise ValueError("Missing columns: " + ", ".join(missing))

omi_mask = train_df["OMI"].eq(1)

hard_neg_mask = (
    train_df["OMI"].eq(0)
    &
    (
        train_df["STEMI"].eq(1)
        |
        train_df["NSTEMI"].eq(1)
    )
)

easy_neg_mask = (
    train_df["OMI"].eq(0)
    &
    ~(
        train_df["STEMI"].eq(1)
        |
        train_df["NSTEMI"].eq(1)
    )
)

n_pos = int(omi_mask.sum())
n_hard = int(hard_neg_mask.sum())
n_easy = int(easy_neg_mask.sum())
n_neg = n_hard + n_easy

print("OMI positives :", n_pos)
print("Hard negatives:", n_hard)
print("Easy negatives:", n_easy)
print("All negatives :", n_neg)
print("Total         :", n_pos + n_neg)

print(
    "Original hard-negative fraction among negatives:",
    round(n_hard / n_neg, 4)
)


OMI positives : 985
Hard negatives: 1309
Easy negatives: 12979
All negatives : 14288
Total         : 15273
Original hard-negative fraction among negatives: 0.0916


In [9]:
# CELL 7 — HARD-NEGATIVE SAMPLER
# Preserves total positive-vs-negative sampling mass.

target_hard_frac = CONFIG["target_hard_negative_fraction"]

if not (0 < target_hard_frac < 1):
    raise ValueError("target_hard_negative_fraction must be between 0 and 1.")

if n_hard == 0 or n_easy == 0:
    raise ValueError("Hard/easy negative group is empty.")

w_pos = 1.0
w_hard = (target_hard_frac * n_neg) / n_hard
w_easy = ((1.0 - target_hard_frac) * n_neg) / n_easy

sample_weights = np.ones(len(train_df), dtype=np.float64)
sample_weights[omi_mask.values] = w_pos
sample_weights[hard_neg_mask.values] = w_hard
sample_weights[easy_neg_mask.values] = w_easy

sample_weights = torch.as_tensor(sample_weights, dtype=torch.double)

generator = torch.Generator()
generator.manual_seed(SEED)

train_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(train_dataset),
    replacement=True,
    generator=generator
)

print("Sampling weights")
print("OMI positive :", round(w_pos, 4))
print("Hard negative:", round(w_hard, 4))
print("Easy negative:", round(w_easy, 4))

positive_mass = n_pos * w_pos
negative_mass = n_hard * w_hard + n_easy * w_easy

print("Expected positive fraction:",
      round(positive_mass / (positive_mass + negative_mass), 4))


Sampling weights
OMI positive : 1.0
Hard negative: 5.4576
Easy negative: 0.5504
Expected positive fraction: 0.0645


In [10]:
# CELL 8 — VERIFY ONE SAMPLED EPOCH

sampled_indices = list(iter(train_sampler))
sampled = train_df.iloc[sampled_indices]

s_pos = int(sampled["OMI"].eq(1).sum())

s_hard = int(
    (
        sampled["OMI"].eq(0)
        &
        (
            sampled["STEMI"].eq(1)
            |
            sampled["NSTEMI"].eq(1)
        )
    ).sum()
)

s_easy = len(sampled) - s_pos - s_hard
s_neg = s_hard + s_easy

print("Sampled epoch composition")
print("OMI positives :", s_pos)
print("Hard negatives:", s_hard)
print("Easy negatives:", s_easy)
print("Positive fraction:", round(s_pos / len(sampled), 4))
print("Hard-neg fraction among negatives:", round(s_hard / s_neg, 4))


Sampled epoch composition
OMI positives : 981
Hard negatives: 7073
Easy negatives: 7219
Positive fraction: 0.0642
Hard-neg fraction among negatives: 0.4949


In [11]:
# CELL 9 — DATALOADERS

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    sampler=train_sampler,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))


Train batches: 955
Validation batches: 168


In [12]:
# CELL 10 — SAME BCE LOSS AS TASK 1

pos_weight_value = n_neg / n_pos

pos_weight = torch.tensor(
    [pos_weight_value],
    dtype=torch.float32,
    device=DEVICE
)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

print("Positive class weight:", round(pos_weight_value, 4))


Positive class weight: 14.5056


In [13]:
# CELL 11 — APPLY SELECTED BASELINE FINE-TUNING MODE

def configure_finetuning(model, mode):

    for p in model.parameters():
        p.requires_grad = False

    for p in model.dense.parameters():
        p.requires_grad = True

    if mode == "last2":

        for stage in model.stage_list[-2:]:
            for p in stage.parameters():
                p.requires_grad = True

    elif mode == "full":

        for p in model.parameters():
            p.requires_grad = True

    else:
        raise ValueError("baseline_mode must be 'last2' or 'full'.")

    return model


model = configure_finetuning(model, CONFIG["baseline_mode"])

print("Baseline mode:", CONFIG["baseline_mode"])
print(
    "Trainable parameters:",
    f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}"
)


Baseline mode: last2
Trainable parameters: 26,709,793


In [14]:
# CELL 12 — OPTIMIZER

def build_optimizer(model, config):

    mode = config["baseline_mode"]

    if mode == "last2":

        backbone_params = []
        head_params = []

        for name, p in model.named_parameters():

            if not p.requires_grad:
                continue

            if name.startswith("dense"):
                head_params.append(p)
            else:
                backbone_params.append(p)

        return torch.optim.AdamW(
            [
                {"params": backbone_params, "lr": config["backbone_lr"]},
                {"params": head_params, "lr": config["head_lr"]}
            ],
            weight_decay=config["weight_decay"]
        )

    if mode == "full":

        early_params = []
        late_params = []
        head_params = []

        late_stage_names = ("stage_list.5", "stage_list.6")

        for name, p in model.named_parameters():

            if not p.requires_grad:
                continue

            if name.startswith("dense"):
                head_params.append(p)

            elif name.startswith(late_stage_names):
                late_params.append(p)

            else:
                early_params.append(p)

        return torch.optim.AdamW(
            [
                {"params": early_params, "lr": config["early_backbone_lr"]},
                {"params": late_params, "lr": config["backbone_lr"]},
                {"params": head_params, "lr": config["head_lr"]}
            ],
            weight_decay=config["weight_decay"]
        )


optimizer = build_optimizer(model, CONFIG)

model_param_ids = {
    id(p) for p in model.parameters() if p.requires_grad
}

optimizer_param_ids = {
    id(p)
    for group in optimizer.param_groups
    for p in group["params"]
}

assert model_param_ids == optimizer_param_ids,     "Optimizer is not connected to the current model."

print("Optimizer correctly attached.")

for i, group in enumerate(optimizer.param_groups):
    n = sum(p.numel() for p in group["params"])
    print(f"Group {i}: lr={group['lr']:.1e} | parameters={n:,}")


Optimizer correctly attached.
Group 0: lr=1.0e-05 | parameters=26,708,768
Group 1: lr=1.0e-04 | parameters=1,025


In [15]:
# CELL 13 — VALIDATION

def evaluate_model(model, loader, criterion, device, threshold=0.5):

    model.eval()

    losses = []
    probabilities = []
    labels = []

    with torch.no_grad():

        for x, y in loader:

            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            logits = model(x).squeeze(1)
            loss = criterion(logits, y)

            probs = torch.sigmoid(logits)

            losses.append(loss.item())
            probabilities.extend(probs.cpu().numpy())
            labels.extend(y.cpu().numpy())

    probabilities = np.asarray(probabilities)
    labels = np.asarray(labels).astype(int)
    predictions = (probabilities >= threshold).astype(int)

    auroc = roc_auc_score(labels, probabilities)
    auprc = average_precision_score(labels, probabilities)
    f1 = f1_score(labels, predictions, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(
        labels,
        predictions,
        labels=[0, 1]
    ).ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) else 0
    specificity = tn / (tn + fp) if (tn + fp) else 0

    return {
        "loss": float(np.mean(losses)),
        "auroc": float(auroc),
        "auprc": float(auprc),
        "f1": float(f1),
        "sensitivity": float(sensitivity),
        "specificity": float(specificity),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "probabilities": probabilities,
        "labels": labels
    }


In [16]:
# CELL 14 — TRAINING

def train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    device,
    config
):

    best_auprc = -1.0
    epochs_without_improvement = 0
    history = []

    mode = config["baseline_mode"]

    save_path = os.path.join(
        config["save_dir"],
        f"task2_ecgfounder_hard_negative_{mode}.pth"
    )

    for epoch in range(1, config["epochs"] + 1):

        model.train()
        train_losses = []

        train_bar = tqdm(
            train_loader,
            desc=f"Epoch {epoch:02d}/{config['epochs']} - Training",
            leave=False
        )

        for x, y in train_bar:

            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            optimizer.zero_grad()

            logits = model(x).squeeze(1)
            loss = criterion(logits, y)

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                config["max_grad_norm"]
            )

            optimizer.step()

            train_losses.append(loss.item())

            train_bar.set_postfix(
                current_loss=f"{loss.item():.4f}",
                avg_loss=f"{np.mean(train_losses):.4f}"
            )

        train_loss = float(np.mean(train_losses))

        metrics = evaluate_model(
            model,
            val_loader,
            criterion,
            device
        )

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": metrics["loss"],
            "auroc": metrics["auroc"],
            "auprc": metrics["auprc"],
            "f1": metrics["f1"],
            "sensitivity": metrics["sensitivity"],
            "specificity": metrics["specificity"]
        })

        print("\n" + "=" * 65)
        print("TASK 2 — HARD-NEGATIVE SAMPLING")
        print("Baseline mode:", mode)
        print(f"Epoch {epoch:02d}/{config['epochs']}")
        print(f"Train Loss : {train_loss:.5f}")
        print(f"Val Loss   : {metrics['loss']:.5f}")
        print(f"AUROC      : {metrics['auroc']:.4f}")
        print(f"AUPRC      : {metrics['auprc']:.4f}")
        print(f"F1         : {metrics['f1']:.4f}")
        print(f"Sensitivity: {metrics['sensitivity']:.4f}")
        print(f"Specificity: {metrics['specificity']:.4f}")
        print(
            f"TN={metrics['tn']} "
            f"FP={metrics['fp']} "
            f"FN={metrics['fn']} "
            f"TP={metrics['tp']}"
        )

        if metrics["auprc"] > best_auprc:

            best_auprc = metrics["auprc"]
            epochs_without_improvement = 0

            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "epoch": epoch,
                    "task": "hard_negative_sampling",
                    "baseline_mode": mode,
                    "val_auprc": metrics["auprc"],
                    "val_auroc": metrics["auroc"],
                    "target_hard_negative_fraction":
                        config["target_hard_negative_fraction"],
                    "config": config
                },
                save_path
            )

            print("\n✓ New best Task-2 model saved")
            print("Path:", save_path)

        else:

            epochs_without_improvement += 1

            print(
                f"\nNo improvement: "
                f"{epochs_without_improvement}/"
                f"{config['patience']}"
            )

            if epochs_without_improvement >= config["patience"]:
                print("\nEarly stopping triggered.")
                break

        print("=" * 65)

    print("\nTraining finished.")
    print("Best validation AUPRC:", round(best_auprc, 4))
    print("Best checkpoint:", save_path)

    return history


In [17]:
# CELL 15 — START TASK 2
# Run only after Task 1 winner is known and CONFIG["baseline_mode"] is set correctly.

history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=DEVICE,
    config=CONFIG
)


Epoch 01/30 - Training:   0%|          | 0/955 [00:00<?, ?it/s]


TASK 2 — HARD-NEGATIVE SAMPLING
Baseline mode: last2
Epoch 01/30
Train Loss : 1.15966
Val Loss   : 0.96656
AUROC      : 0.8508
AUPRC      : 0.3336
F1         : 0.3480
Sensitivity: 0.5241
Specificity: 0.9019
TN=2272 FP=247 FN=79 TP=87

✓ New best Task-2 model saved
Path: /content/drive/MyDrive/ECG_Project/OMI_model/task2_hard_negative/task2_ecgfounder_hard_negative_last2.pth


Epoch 02/30 - Training:   0%|          | 0/955 [00:00<?, ?it/s]


TASK 2 — HARD-NEGATIVE SAMPLING
Baseline mode: last2
Epoch 02/30
Train Loss : 1.04027
Val Loss   : 0.99840
AUROC      : 0.8583
AUPRC      : 0.3529
F1         : 0.3644
Sensitivity: 0.4940
Specificity: 0.9198
TN=2317 FP=202 FN=84 TP=82

✓ New best Task-2 model saved
Path: /content/drive/MyDrive/ECG_Project/OMI_model/task2_hard_negative/task2_ecgfounder_hard_negative_last2.pth


Epoch 03/30 - Training:   0%|          | 0/955 [00:00<?, ?it/s]

...


TASK 2 — HARD-NEGATIVE SAMPLING
Baseline mode: last2
Epoch 05/30
Train Loss : 0.95594
Val Loss   : 0.95271
AUROC      : 0.8618
AUPRC      : 0.3338
F1         : 0.3623
Sensitivity: 0.5904
Specificity: 0.8900
TN=2242 FP=277 FN=68 TP=98

No improvement: 3/5


Epoch 06/30 - Training:   0%|          | 0/955 [00:00<?, ?it/s]


TASK 2 — HARD-NEGATIVE SAMPLING
Baseline mode: last2
Epoch 06/30
Train Loss : 0.88461
Val Loss   : 1.08113
AUROC      : 0.8622
AUPRC      : 0.3320
F1         : 0.3895
Sensitivity: 0.4940
Specificity: 0.9313
TN=2346 FP=173 FN=84 TP=82

No improvement: 4/5


Epoch 07/30 - Training:   0%|          | 0/955 [00:00<?, ?it/s]


TASK 2 — HARD-NEGATIVE SAMPLING
Baseline mode: last2
Epoch 07/30
Train Loss : 0.85299
Val Loss   : 1.07016
AUROC      : 0.8622
AUPRC      : 0.3064
F1         : 0.3893
Sensitivity: 0.5241
Specificity: 0.9230
TN=2325 FP=194 FN=79 TP=87

No improvement: 5/5

Early stopping triggered.

Training finished.
Best validation AUPRC: 0.3529
Best checkpoint: /content/drive/MyDrive/ECG_Project/OMI_model/task2_hard_negative/task2_ecgfounder_hard_negative_last2.pth


## Clean ablation rule

For Task 2, keep unchanged:
- preprocessing
- patient split
- architecture
- selected fine-tuning mode
- learning rates
- optimizer
- `pos_weight`
- batch size
- early stopping
- checkpoint selection by validation AUPRC

The intended experimental change is only the negative sampling distribution.
